# RLVR-Decomposed Training on Google Colab

This notebook sets up and runs RLVR-Decomposed training on Google Colab.

**Important**: Make sure you're using a GPU runtime!
- Go to Runtime → Change runtime type → Hardware accelerator → GPU

## Step 1: Check GPU Availability

In [ ]:
!nvidia-smi

# Check which GPU you have
import subprocess
gpu_info = subprocess.check_output('nvidia-smi --query-gpu=name,memory.total --format=csv', shell=True).decode('utf-8')
print("\n" + "="*50)
print("Your GPU:")
print("="*50)
print(gpu_info)

if 'T4' in gpu_info:
    print("\n⚠️  You have a T4 GPU (Colab Free). Use the FREE tier script.")
    print("💡 Recommend using Qwen2.5-0.5B or Qwen2.5-1.5B model for stability.")
elif 'A100' in gpu_info or 'V100' in gpu_info:
    print("\n✅ You have a high-end GPU! Use the PRO tier script.")
    print("💡 Qwen3-4B should work well.")
else:
    print("\n🤔 Unknown GPU. Try the FREE tier script first.")

## Step 2: Clone Repository with Colab Support Files

In [ ]:
# Clone the repository and checkout the Colab-ready branch
!git clone https://github.com/sectumsemprra/RLVR-Decomposed.git
%cd RLVR-Decomposed

# Checkout the branch with Colab support
!git checkout claude/check-qwen3-dependencies-W4BzB

# Verify the Colab files exist
import os
required_files = [
    'requirements_colab.txt',
    'run_qwen3-4b_psr_nsr_colab_free.sh',
    'run_qwen3-4b_psr_nsr_colab_pro.sh',
    'COLAB_SETUP_GUIDE.md'
]

print("\nChecking for required files:")
all_found = True
for f in required_files:
    exists = os.path.exists(f)
    status = "✅" if exists else "❌"
    print(f"{status} {f}")
    if not exists:
        all_found = False

if all_found:
    print("\n✅ All required files found!")
else:
    print("\n❌ Some files are missing. Make sure you're on the correct branch.")

## Step 3: Install Dependencies

This will take 5-10 minutes.

In [ ]:
# Install main dependencies
!pip install -r requirements_colab.txt

## Step 4: Install Flash Attention (Optional but Recommended)

This takes 10-15 minutes. You can skip this if you want to start faster, but training will be slower.

In [ ]:
# Install flash-attention (this takes time)
# Uncomment the next line if you want flash-attn (recommended for better performance)
# !python -m pip install flash-attn --no-build-isolation

print("⏭️  Skipped flash-attn installation. Uncomment the line above to install it.")

## Step 5: Verify Installation

In [ ]:
# Check versions
import transformers
import tensordict
import ray

print("Package Versions:")
print("=" * 50)
print(f"Transformers: {transformers.__version__} (expected: 4.52.2)")
print(f"TensorDict: {tensordict.__version__} (expected: 0.7.2)")
print(f"Ray: {ray.__version__} (expected: >=2.10)")

try:
    import vllm
    print(f"vLLM: {vllm.__version__} (expected: 0.8.5)")
except Exception as e:
    print(f"vLLM: Error - {e}")

print("\n✅ Installation verification complete!")

## Step 6: Setup Weights & Biases (Optional)

This is for experiment tracking. You can skip this if you don't want to track experiments.

In [ ]:
# Login to wandb (optional)
import wandb
wandb.login()

# Or skip wandb by commenting it out in the training script

## Step 7: Prepare Data

⚠️ **Important**: You need to have these data files:
- `./data/math/train.parquet`
- `./data/math/test.parquet`
- `./data/aime2025/test.parquet`
- `./data/amc23/test.parquet`

If you don't have them, you'll need to download or generate them.

In [ ]:
# Check if data files exist
import os

data_files = [
    './data/math/train.parquet',
    './data/math/test.parquet',
    './data/aime2025/test.parquet',
    './data/amc23/test.parquet'
]

print("Checking data files:")
for f in data_files:
    exists = os.path.exists(f)
    status = "✅" if exists else "❌"
    print(f"{status} {f}")

# If you need to download data, add your download commands here
# Example:
# !mkdir -p data/math data/aime2025 data/amc23
# !wget -O data/math/train.parquet YOUR_DATA_URL

## Step 8: Choose Configuration

Edit the script based on your GPU tier and preferences.

In [ ]:
# For COLAB FREE (T4 GPU):
# Recommend changing model to smaller one for stability

# Read the free tier script
with open('run_qwen3-4b_psr_nsr_colab_free.sh', 'r') as f:
    script = f.read()

# Replace model with smaller one (recommended for T4)
script = script.replace(
    'model_name=Qwen/Qwen3-4B  # Original, will be tight on memory',
    'model_name=Qwen/Qwen2.5-0.5B-Instruct  # Smaller model, more stable on T4'
)

# Save modified script
with open('run_qwen3-4b_psr_nsr_colab_free_modified.sh', 'w') as f:
    f.write(script)

!chmod +x run_qwen3-4b_psr_nsr_colab_free_modified.sh

print("✅ Created modified script with smaller model for better stability on T4!")
print("\nYou can now run:")
print("  - For FREE tier (T4): run_qwen3-4b_psr_nsr_colab_free_modified.sh")
print("  - For PRO tier (A100/V100): run_qwen3-4b_psr_nsr_colab_pro.sh")

## Step 9: Run Training

Choose ONE of the cells below based on your GPU tier.

In [ ]:
# FOR COLAB FREE TIER (T4 GPU)
# This uses the modified script with a smaller model
!bash run_qwen3-4b_psr_nsr_colab_free_modified.sh

In [ ]:
# FOR COLAB PRO/PRO+ (A100/V100)
!bash run_qwen3-4b_psr_nsr_colab_pro.sh

## Step 10: Monitor Training

Training will output progress in the cell above.
You can also monitor on Weights & Biases if you set it up.

## Troubleshooting

### Out of Memory (OOM)
If you get OOM errors:
1. Use an even smaller model (Qwen2.5-0.5B-Instruct)
2. Reduce batch size in the script
3. Reduce token limits

### Script Freezes
Try reducing `num_cpus` or check the troubleshooting section in COLAB_SETUP_GUIDE.md

### Version Conflicts
```python
!pip uninstall -y vllm transformers tensordict
!pip install vllm==0.8.5 transformers==4.52.2 tensordict==0.7.2
```

### Need More Help?
Check the comprehensive guide:
```python
!cat COLAB_SETUP_GUIDE.md
```

## Save Checkpoints to Google Drive

Run this to save your trained models to Google Drive so they don't get lost when Colab disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Copy checkpoints to Drive
# Adjust the path based on where checkpoints are saved
# !cp -r ./checkpoints /content/drive/MyDrive/rlvr_checkpoints/

print("Remember to copy your checkpoints to Google Drive!")